
# Dimensionality Reduction and SVD

## Prerequisities
- Eigen values, Eigen vectors, Linear algebra, Matrix decomposition

## Learning Objectives

After reading this notebook, students should be able to:

- Relate dimensions with attributes, training data, accuracy.
- discuss curse of dimensionality and necessity for dimensionality reduction.
- Understand Singular Value Decomposition (SVD) as a low-rank decomposition for any matrices.

## Curse of Dimensionality


In machine learning, data set with large number of attributes is referred as a high dimensional data. Increase in dimension exponential increases the amount of data required to train a generalizable model.

 For example, if you need 5 points to cover a one-dimensional line, two-dimensional cover areas, you require 25 points. If we proceed and try to fill three-dimensional volumes, you need 125 points to cover the whole volume. As the dimension increases, the point or data it takes to fill it increases exponentially.



 <figure align="center">
       <!-- <img src="https://drive.google.com/uc?id=1twJ8lu9DSF6Bl5GVJ3ua4TRROyt0SQ-C" height="200" width="600"> -->
       <img src= "https://i.postimg.cc/V6xnvKZJ/image.png" height="200" width="600">
       <figcaption>Figure 1: Simple Linear Regression </figcaption>
   </figure>


Here, increasing the dimension from $1$ to $3$ is analogous to increasing the attributes in a dataset. Also, the increase in points with the increase in dimension is analogous to an increase in training data. With the increase in attributes, the number of training data to generalize the model also increases exponentially.

__Note__: In Linear Regression unit, there was an assumption that the number of observations must be higher than the number of unknowns. The concept behind this assumption is the same, as we explained in the above section. The number of unknowns(including the intercept) equals the number of attributes in the dataset. Increment in unknowns implies an increase in attributes that imply an increase in dimensions. With the increase in dimensions, observations need to increase.

To demonstrate the curse of dimensionality we will use house prediction dataset which is a linear regression problem.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
dataset = pd.read_csv('https://storage.googleapis.com/codehub-data/1-lv2-10-1-house_prediction.csv')

X = dataset.drop('Output', axis=1).values
y = dataset['Output'].values.reshape(X.shape[0],1)
dataset.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,...,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,Output
0,0,-0.595633,0.656854,-0.684699,-0.673823,1.887610,-0.955794,0.244962,0.727445,0.952900,0.508535,-1.159876,-0.792027,0.247131,0.247016,0.216332,0.030032,0.987333,-1.059502,0.349696,-0.012446,-0.717748,-0.287227,-0.593095,-1.137787,0.789334,1.988265,0.230974,-0.922095,-0.580896,-0.023586,-1.140167,0.757290,1.429268,-0.317406,0.312277,1.033168,-0.436566,0.864269,-0.159033,...,0.173967,-1.874588,0.088016,0.384914,-0.784461,-0.416663,0.546173,-0.593291,0.406499,-0.767546,-0.815956,-2.574882,0.103371,-1.063059,-0.578222,0.324676,-0.898555,-0.899417,1.446544,0.387430,-0.884279,-0.178496,-0.589442,0.627858,0.178913,-0.679499,-0.310437,0.523817,-1.412448,-1.736277,-1.899894,-0.824319,0.343274,0.262927,-0.976216,0.326459,-0.572893,-0.082047,-0.379181,-161.154360
1,1,-0.514398,-0.799939,0.023570,0.121426,0.771489,-0.302288,1.372522,0.499081,0.543479,1.732247,0.704477,-1.187009,-2.158765,-0.357000,0.502073,0.044530,-0.090989,1.006248,-0.199008,0.238012,0.658693,-0.473608,-0.903455,-0.562130,1.148534,-2.222780,-0.557287,0.112351,-2.230489,-0.579181,0.185699,-0.147535,0.228886,0.643768,0.436643,-0.232190,0.609582,0.731239,-1.463704,...,-0.332766,0.777479,-1.852284,1.083566,-1.119386,0.604615,-0.523609,-1.076232,-0.247533,1.162202,-0.047411,0.682684,-0.873031,0.983497,1.831192,1.128899,0.250550,-0.356416,-1.341162,-0.447846,1.512808,-1.064790,-1.832698,-0.825434,-0.682514,-0.169681,0.440522,1.085185,-0.971717,0.663359,-0.386877,1.156375,-0.254059,1.706383,-0.498046,0.225324,-0.925442,-0.338938,-2.701207,268.929554
2,2,0.050114,2.101232,1.302136,1.090586,-0.382401,-0.857586,-1.228514,-0.081406,2.054224,0.451998,-1.116006,-0.523342,-0.846446,-0.131473,1.090225,1.034471,0.143985,-0.430431,-0.186952,0.346643,-0.133094,1.147636,0.518908,-1.059241,-0.341139,0.123335,2.245490,0.158724,0.474971,-1.481409,1.645622,1.493403,-0.582553,-1.741065,-2.795273,-0.607317,-0.847070,-1.584085,2.092329,...,-1.260051,0.867603,-0.840392,-0.337937,1.497425,0.262202,-0.583670,-0.466383,-0.818737,-0.032398,-0.316605,-0.412552,0.443565,0.417756,-0.083269,-0.775599,-0.774057,0.539785,-0.183384,0.684940,-0.807087,0.964988,-0.017666,-0.499472,-1.220820,1.483993,2.012098,-0.630836,0.698952,0.629735,0.884945,-0.181102,-0.711264,0.378821,-0.218168,0.523393,-0.643925,-2.892352,-1.756088,265.928296
3,3,1.703842,-0.228698,-0.966759,0.301334,0.905673,0.150270,-1.099488,-1.448912,0.483969,0.709805,-0.212260,-0.068940,0.464470,0.918498,-0.325664,0.050301,1.050292,-0.189306,-0.590692,-1.505414,0.300889,-0.606685,-1.028367,0.263434,0.348522,-1.062493,0.475954,-1.047136,0.793541,-1.014798,-0.564438,-0.281678,-1.830695,-1.098698,0.336950,0.855404,-0.117167,-1.225571,-0.939845,...,1.942689,0.083703,-1.640842,-0.310490,-0.812403,-1.266038,-0.645085,-0.259582,-1.006015,0.204755,-0.888034,-0.354004,0.149828,1.277788,0.502101,-2.129159,0.653659,-0.336465,0.013366,-0.517456,0.134203,0.113456,-0.394730,-0.659126,2.208726,0.164566,-0.224012,1.977970,-0.748762,0.240861,-0.300816,0.056597,-1.137565,-0.601429,0.998263,1.517622,0.603316,-0.837909,-0.210869,-189.920382
4,4,1.030185,-1.365925,-0.642969,2.219637,-0.066616,0.885881,-1.194113,-2.565285,-0.198834,-0.149267,0.870191,-1.423889,2.410330,0.544874,-0.264045,1.123286,-0.431374,-0.386184,0.002364,-0.652925,-1.177159,-0.441727,0.087609,-1.079974,1.123924,0.595818,0.122881,1.688104,0.760549,0.355218,0.237381,-1.852088,0.379770,-0.619508,0.835377,-0.618425,-0.514621,0.723639,0.209328,...,-0.567220,0.946817,0.680136,-0.034282,0.880781,0.978971,-1.613498,-0.046083,2.307548,0.770824,0.574859,-0.356441,1.168652,-0.078523,0.907408,-1.714256,0.103520,1.406853,0.044346,-1.574338,-1.040065,1.171492,0.095999,0.811146,-1.583289,-1.979380,-0.282012,0.515447,-1.084214,-0.334073,1.135425,0.528185,-1.893242,0.8762

This is a common hosehold price estimation dataset for linear regression. The dataset has $100$ features and $1$ output. Features are already scaled.
For simplicity, let's compare this regression problem to the house price prediction problem. For this regression problem, we will see five different models. Five different models imply five different sets of inputs.


Before creating the models, let's split the dataset into train and test sets. We will train regression models using train set and test the performance on test set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

## Changing into pandas dataframes
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)

For the first model, we will take $10$ features and train the linear regression model with a dataset comprising $10$ features and single output.  



In [ ]:
X_first_train = X_train.iloc[:, :10]
X_first_test = X_test.iloc[:, :10]

Since this is a regression problem, we use adjusted $R^2$ to evaluate the model. Let's first write a function `adjusted_r2_score` which calculates the adjusted R squared score.


In [ ]:
from sklearn.metrics import r2_score

def adjusted_r2_score(y_test, y_pred):
    n = X.shape[0]
    d = X.shape[1]

    r2 = r2_score(y_test, y_pred)
    adj_r2 = 1-(1-r2)*(n-1)/(n-d-1)

    return adj_r2

Now, we will train the first model with `X_first_train` and test the model with `X_first_test` with the help of adjusted R squared score.

In [ ]:
from sklearn.linear_model import LinearRegression

def train_and_evaluate(X_train_model, X_test_model):

    linear_regression = LinearRegression()
    linear_regression.fit(X_train_model, y_train)
    y_pred = linear_regression.predict(X_test_model)
    adj_r2 = adjusted_r2_score(y_test, y_pred)

    return adj_r2

In [ ]:
adjusted_r2 = []
adjusted_r2_first = train_and_evaluate(X_first_train, X_first_test)
adjusted_r2.append(adjusted_r2_first)
adjusted_r2_first

-0.0012348225867768736

For this model with ten inputs, let our adjusted $R^2$ score be $\bar{R}^2_1$ which is $-0.001234$.


Similarly, we take other four different number of inputs and evaluated the models. Now, we will plot the adjusted $R^2$ for each of the five models and explain the effect of dimension on performance.


 <figure align="center">
       <!-- <img src="https://drive.google.com/uc?id=1PIG6Q81j00-5V5ESmFNppyAFV0h67XDi" height="400" width="600"> -->
      <img src="https://i.postimg.cc/fW6QWYw1/image.png" height="400" width="600">
       <figcaption>Figure 2: Dimension and performance </figcaption>
   </figure>






For the second model, we take $40$ inputs. Here, the total attributes are $41$, including a response. Here, this model has more information provided by more input variables.

From the Figure 2, the adjusted $R^2$ score for the second model is $\bar{R}^2_2$ which is $0.235498$. Since now we have two different models for the same regression problem, we can always compare them. While comparing:

$$\bar{R}^2_1 <\bar{R}^2_2 $$




Hence, the second model is the preferred one. It's performing better since the information provided is more. Now we will increase the number of features in the third model from $40$ to $63$, whoose adjusted $R^2$  is represented as $\bar{R}_3^2$ which is $0.46026$. Comparing third model with the second model, we get:




$$\bar{R}^2_2 <\bar{R}^2_3 $$


The fourth model has $65$ features or inputs.
Adjusted $R^2$ is $\bar{R}^2_4$ whoose value if $0.46024$. Comparing this model with the earlier third model, we get the result as:


$$\bar{R}^2_3 >\bar{R}^2_4$$

Here's a catch! Increment in features here couldn't increase the adjusted $R^2$ or performance of the model. Instead, we are facing a decrease in adjusted $R^2$. Increment in information through the addition of features is degrading the performance. Why is this happening? Before answering the question, let's briefly discuss fifth model, where we will use $70$ features. $70$ features mean $70$ different kinds of information to predict the price of the house.

$\bar{R}_5^2$ is the adjusted $R^2$ score for the fifth model which is $0.46020$ then:

$$\bar{R}^2_4 >\bar{R}^2_5  $$

Increasing features is not helping us to improve the performance. The overall adjusted $R^2$ for all five models can be presented as:

$$\bar{R}^2_1<\bar{R}^2_2<\bar{R}^2_3>\bar{R}^2_4> \bar{R}^2_5 $$








Hence, the third model with $63$ features is the best model among all five with the highest adjusted $R^2$. We will answer the question, _Why our model degrade its performance while transitioning to fourth model from third model ?_

 When we increase the features from $10$ to $40$ to $63$, the performance of the model increases due to the increment in the number of information. We expect the same after we increase the number of features to $65$, but as the number of features increased to $65$, the model decreased its performance. This is because there is a threshold for the number of features or dimensions that determine the performance of the model according to the available features. For the house prediction regression model, the threshold is $63$ features. If we provide more than $63$ features to the model, the model becomes confused. Or in simple words, the added features to the model are not required for the model. These added features above the threshold instead make the model confused and deviates the model from the right prediction. Also, the training data should increase exponentially as you increase the features. The model becomes complex with the addition of features, and there is a high chance of overfitting. So, one must be very careful while increasing the features or dimensions for any machine learning models.



Features are boon to the algorithm; they serve as the source of information. But a huge number of features often proves to be a curse. This is often termed as  __the curse of dimensionality__.

## Dimensionality Reduction: Needs and Techniques

Now, before jumping into dimensionality reduction techniques, let's first see: Why do we need to reduce the dimension?

Let's suppose you are dealing with the multiclass classification problem to classify the geographical regions with the available features. Among all available features here are few of the features listed:

- $x_1$: _Number of skidding accidents_
- $x_2$: _Number of burst water pipes_
- $x_3$: _Number of patients with heat strokes_
- $x_4$: _Snow-plow expenditures_

These are just a few of the attributes of the dataset. These attributes seem like completely different attributes that tell us different things, but if we look at these features closely, we will realize that there is a single factor that probably explains most of these features, i.e., _temperature_. You won't skid unless you have ice. Similarly, your pipes won't burst unless you have an extreme cold atmosphere, and the rest of the two features are also closely related to the temperature. So, we can exclude these four features with the inclusion of a single feature, _temperature_. In this way, we reduced the four-dimensional features to one dimensional.


We can relate this toy example to the real Machine Learning examples such as Image processing or text processing. Image data are high dimensional. We have hundreds of thousands of pixels, and if we are dealing with text, we have millions to billions of words. That's the way data comes in, but in reality, the true dimension of data might be a lot lower. As, we always tend to find this true dimension of the dataset, which is always lower than the original dataset. This is _Dimensionality Reduction_. Some of the necessities behind dimensionality reduction are listed below:

- The number of training data lowers with the decrease in dimension. This way, the space required to store the data also decreases, which is computationally easier to handle.

- Training time or computational time also decreases with the decrease in training data.

- With dimensionality reduction, we can remove the redundant and highly correlated features. An illustration of this is shown in the preceding example.

- Some machine learning algorithms don't perform well in high dimensions with many features. Reducing the dimension can make the model perform better, with high accuracy and less error margin.


And there are other advantages of reducing the dimensions that deal with visualization simplicity, etc.


## Dimnesional Reduction techniques

Now that we know the importances of dimensionality reduction, we will see some of the dimensionality reduction techniques. Mainly, dimensionality reduction techniques are classified as:



- Linear Dimensionality Reduction Techniques

    These linear dimensionality reduction techniques analyse high dimensional data with the help of data features, such as covariance, dynamical structure, correlation between dataset, input-ouput relationships and margin between data classes. Some of the linear dimensionality reduction techniques are: Principal Componenet Analysis (PCA), Singular Value Decomposition (SVD), Linear Discriminant Analysis (LDA).

    Principal Component Analysis(PCA) and Singular Value Decomposition(SVD) are linear dimensionality reduction techniques that that performs dimensionality reduction by embedding the data into a linear subspace of lower dimensionality.They construct a low-dimensional representation of the data that describes as much of the variance. We will cover SVD in this notebook whereas PCA will be covered in the next notebook.






- Non Linear Dimensionality Reduction Techniques

    In contrast to the linear techniques, these non linear techniques have the ability to deal with non linear complex data. Since most of the real world data  are likely to form a high non linear structure and requires non linear hyperplane, this technique offers an advantage. Also, non linear techniques outperform their linear counterparts on complex artificial tasks.

  For example: Swiss roll dataset comprises a set of points that lie on a spiral-like two-dimensional manifold which is embedded
within a three-dimensional space. Nonlinear dimensional reduction techniques are perfectly able to find this
embedding, whereas linear techniques fail to do so. In contrast to these successes on artificial datasets,
successful applications of nonlinear dimensionality reduction techniques on natural datasets are less
convincing. Some of the non linear dimensionality reduction techniques are: Kernel PCA, Isomap, Maximum Variance Unfolding, diffusion maps, multilayer encoders, etc.





## Singular Value Decomposition (SVD)

Let's understand Singular Value Decomposition (SVD) with a simple example. Consider a matrix $\mathbf{X}$ whose rows corresponds to the vectors: $\mathbf{a}$ and $\mathbf{b}$.

$$\mathbf{X} =
\begin{pmatrix}
a_x & a_y \\
b_x & b_y \\
\end{pmatrix}$$


Consider another matrix $\mathbf{V}$ whose columns corresponds to two orthonormal vectors: $\mathbf{v_1}$ and $\mathbf{v_2}$.

$$\mathbf{V} =
\begin{pmatrix}
v_{1x} & v_{2x} \\
v_{1y} & v_{2y} \\
\end{pmatrix}$$


**Our goal here is to project the vectors present in the rows of the matrix $\mathbf{X}$ onto the vectors present in the columns of the matrix $\mathbf{V}$.**

<figure align="center">
       <!-- <img src="https://drive.google.com/uc?id=1sRckOQgZEl1ycnZjM26dcZFusBPiMsXz"> -->
        <img src="https://i.postimg.cc/5ypzgfNC/image.png">

       <figcaption>Figure: Projection of $\mathbf{a}$ and $\mathbf{b}$ onto $\mathbf{v_1}$ and $\mathbf{v_2}$</figcaption>
   </figure>


The figure above shows the projections of vectors $\mathbf{a}$ and $\mathbf{b}$ onto the vectors $\mathbf{v_1}$ and $\mathbf{v_2}$.

Here,

$S_{a1}$ is the length of the projection $p_{a1}$ of $\mathbf{a}$ onto $\mathbf{v_1}$.

$S_{a2}$ is the length of the projection $p_{a2}$ of $\mathbf{a}$ onto $\mathbf{v_2}$

$S_{b1}$ is the length of the projection $p_{b1}$ of $\mathbf{b}$ onto $\mathbf{v_1}$

$S_{b2}$ is the length of the projection $p_{b2}$ of $\mathbf{b}$ onto $\mathbf{v_2}$


The length of the projection of a vector onto a unit vector can be calculated using their dot product. However, with the matrices we have, the length of projections of vectors $\mathbf{a}$ and $\mathbf{b}$ on vectors ${\mathbf{v_1}}$ and $\mathbf{v_2}$ can be calculated simultaneously using the multiplication of matrix $\mathbf{X}$ and $\mathbf{V}$ *, i.e.,*

$$\mathbf{X}.\mathbf{V} = \begin{pmatrix}
a_x & a_y \\
b_x & b_y \\
\end{pmatrix} \times \begin{pmatrix}
v_{1x} & v_{2x} \\
v_{1y} & v_{2y} \\
\end{pmatrix} = \begin{pmatrix}
S_{a1} & S_{a2} \\
S_{b1} & S_{b2}
\end{pmatrix} = \mathbf{S} \dots \dots(1)$$

where, $\mathbf{S}$ is the matrix containing the length of projections of $\mathbf{a}$ and $\mathbf{b}$ onto $\mathbf{v_1}$ and $\mathbf{v_2}$. The first column contains the length of projections on $\mathbf{v_1}$ and the second column contains the length of projections on $\mathbf{v_2}$.

We can write $(1)$ as:

$$\mathbf{X} = \mathbf{S} \mathbf{V}^{-1}$$

$$$$

Since $\mathbf{V}$ is an orthonormal vector, $\mathbf{V}^{-1} = \mathbf{V}^T$. So

$$\mathbf{X} = \mathbf{S} \mathbf{V}^{T}\dots \dots (2)$$



To normalize the columns of $\mathbf{S}$ to be unit vectors, we need to divide each column vectors by their magnitudes. After dividing the columns by their magnitudes, we need to multiply the columns of $\mathbf{S}$ with their magnitude to preserve the equality.

Let $\sigma_1$ and $\sigma_2$ be the magnitudes of first and second column respectively then $\mathbf{S}$ can be decomposed as:

\begin{align}
\mathbf{S}&=   \begin{pmatrix}
\frac{S_{a1}}{\sigma_1} & \frac{S_{a2}}{\sigma_2} \\
\frac{S_{b1}}{\sigma_1} & \frac{S_{b2}}{\sigma_2}
\end{pmatrix} \times \begin{pmatrix}
\sigma_1 & 0 \\
0 & \sigma_2
\end{pmatrix}\\
\\
&= \begin{pmatrix}
U_{a1} & U_{a2} \\
U_{b1} & U_{b2}
\end{pmatrix} \times \begin{pmatrix}
\sigma_1 & 0 \\
0 & \sigma_2
\end{pmatrix}\\
\\
\mathbf{S} &=\hspace{1.25cm}\mathbf{U} \hspace{1.3cm}\times \hspace{1.3cm}\mathbf{\Sigma} \dots \dots (3)
\end{align}

From (2) and (3), we get:

$$\therefore \mathbf{X} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^{T}$$

This is the expression for Singular Value Decomposition or SVD.


Hence, any real matrix, $\mathbf{X}$ with dimension $\text{m} \times \text{n}$ can be decomposed as:

$$\mathbf{X}=\mathbf{U}\mathbf{Σ}\mathbf{V}^{T}$$

where,

- $\mathbf{X}$ is the matrix of observations.


- $\mathbf{U}$ is an $\text{m}×\text{m}$ column-orthonormal matrix; that is, each of its columns is a unit vector, and the dot product of any two columns is $0$.

- $\mathbf{V}$ is an $\text{n}×\text{n}$ column-orthonormal matrix. Note that we always use $\mathbf{V}$ in its transposed form, so it is the rows of $\mathbf{V}^{T}$ that are orthonormal.

- $\mathbf{Σ}$ is an $\text{m}×\text{n}$ diagonal matrix; zero entries everywhere, all except on the leading diagonal with elements arranged in descending order of magnitude.  The elements of $\mathbf{Σ}$ are called the singular values of $\mathbf{X}$.




## Relation of SVD to Eigen values and Eigen vectors
Now, we will see how we calculate all these terms.


Multiplying $\mathbf{X}$ by its transpose gives us:



$$
\begin{matrix}
\mathbf{X}^T\mathbf{X}  & = (\mathbf{U}\mathbf{Σ}\mathbf{V}^T)^T(\mathbf{U}\mathbf{Σ}\mathbf{V}^T)\\
\
 &  = (\mathbf{V}\mathbf{Σ}^T\mathbf{U}^T)(\mathbf{U}\mathbf{Σ}\mathbf{V}^T)
\end{matrix}
$$

$$\ \ \ \ \ \ = \mathbf{V}\mathbf{Σ}^T\mathbf{U}^T\mathbf{U}\mathbf{Σ}\mathbf{V}^T \tag{1}$$

$\mathbf{U}$ is a orthonormal matrix. From the condition of orthonormality:

$$\mathbf{U}^T\mathbf{U} = \mathbf{I}\ (\text{Identity Matrix})$$

And for the diagonal matrix $\mathbf{Σ}$, $\mathbf{Σ}^T\mathbf{Σ} = \mathbf{Σ}^2$. So equation $\text{(1)}$ can be written as:

$$\mathbf{X}^T\mathbf{X} = \mathbf{V}\mathbf{Σ}^2\mathbf{V}^T \tag{2} $$


Now, we should keep in mind that $\mathbf{X}^T\mathbf{X}$ is a symmetric positive definite matrix. There is an eigenvector-eigenvalue factorization for positive definite matrices as:

$$\mathbf{X}^T\mathbf{X} = \mathbf{Q}\mathbf{\wedge}\mathbf{Q}^T\tag{3}$$

Here, $\mathbf{\wedge}$ is a diagonal matrix with diagonal elements as eigenvalues of $\mathbf{X}^T\mathbf{X}$. $\mathbf{Q}$ is the matrix of eigenvectors, which are orthonormal since the eigenvectors of positive definite matrix are orthogonal and can be made orthonormal.


So, from equation $\text(2)$ and $\text(3)$, the columns of $\mathbf{V}$ are the eigenvectors of $\mathbf{X}^T\mathbf{X}$, not of $\mathbf{X}$. The diagonal elements (singular values) of  diagonal (but rectangular) matrix $\mathbf{Σ}$ are square root of eigenvalues from  $\mathbf{X}^T\mathbf{X}$ or $\mathbf{X}\mathbf{X}^T$. Those
positive entries on the diagonal positions are the singular values of $\mathbf{X}$. They fill the first $\text{r}$ places on the main diagonal of $\mathbf{Σ}$—when $\mathbf{X}$ has rank $\text{r}$. The rest of $\mathbf{Σ}$ is $0$.







Similarly, $\mathbf{U}$ must be an eigenvector matrix of $\mathbf{X}\mathbf{X}^T$. Eigenvalues of $\mathbf{X}\mathbf{X}^T$ and  $\mathbf{X}^T\mathbf{X}$ are the same, so you can find out eigenvalues from any of these two and use it to determine singular values of $\mathbf{X}$.

With this, we can use Singuar Value Decomposition (SVD) to decompose the matrix as:


$$\mathbf{X}=\mathbf{U}\mathbf{Σ}\mathbf{V}^{T} =\text{(orthonormal)(diagonal)(orthonormal)}$$


From the derivation above, we know:

- Columns of $\mathbf{U}$ of dimenion($\text{m}×\text{m}$) are eigenvectors of $\mathbf{X}\mathbf{X}^T$.
- Columns of $\mathbf{V}$ of dimension($\text{n}×\text{n}$)  are eigenvectors of $\mathbf{X}^T\mathbf{X}$ .


The $\text{r}$ singular values on the diagonal of $\mathbf{Σ}$, ($\text{m}×\text{n}$) are the square roots of the nonzero eigenvalues of both $\mathbf{X}\mathbf{X}^T$ and $\mathbf{X}^T\mathbf{X}$.


We can now use this decomposition to lower the dimension by reducing the rank with the removal of singular values of negligible magnitudes. We will see a detail application with code in the programming material notebook. For now, let's get some theoretical insights.

# SVD for dimensionality reduction


Singular value decomposition is a very useful tool for dimensionality reduction.Dimensionality reduction can be achieved by simply dropping columns of $\mathbf{U}$ and rows of $\mathbf{V}^T$. So, we can avoid or not use the columns below a certain number. Doing this helps in reducing dimensionality as we are not taking all of the available components. On the other hand, doing this also reduces the accuracy of our representation. So, there is a trade-off here. Often, we can ignore the last few singular values without significant deviation from the original matrix.

<figure align="center">
       <!-- <img src="https://drive.google.com/uc?id=1NQCvMKiuJi7r_gimwD-cF_pdCICMu_3K" height="400" width="650"> -->
         <img src="https://i.postimg.cc/Z5PSM0z2/image.png" height="400" width="650">
       <figcaption>Figure 2: Dimensionality Reduction with SVD</figcaption>
   </figure>



The dimension is equal to the rank. The original matrix has $\text{r}$ rank or dimension, and we reduced the rank or dimension to $\text{k}$ by removing all singular values other than the highest $\text{k}$ singular values (all other singular values are made 0). Reducing the rank simultaneously reduces the number of columns vectors of $\mathbf{U}$ and $\mathbf{V}$. Hence, the dimensionality of $ \mathbf{X}$ is reduced by taking only the components that have high enough variance to fully represent the original matrix or data.

## Key Takeaway

- More features or attributes imply a high dimension, which ultimately means more information to the model.
- Excessive information to the model than required often makes the model confused. This confusion can lead to low accuracy and high error to the model which is __Curse of Dimensionality__.
- There are many dimensionality reduction techniques which can be broadly classified into linear and non linear techniques.
- Singular Value Decomposition (SVD)is one of the most popular linear dimensionality reduction techniques that is primarily used as matrix decomposition technique.